# Day 026 — Exercise 5: ai_generate_section

**What you'll build:** `ai_generate_section(topic, data_snippet, model)` — calls the LLM with `format='json'` to generate a `{heading, body}` dict from raw data text.

**Why it matters:** This is the bridge between raw data and a human-readable report section. The schema-guided approach (Day 4) gives you structured output you can pass directly to `create_pdf_report` or `create_docx_report`. The `json.loads` + fallback pattern ensures you always get a usable dict even if the LLM returns slightly malformed JSON.

In [ ]:
import ollama
import json

## Your Implementation

In [ ]:
def ai_generate_section(
    topic: str,
    data_snippet: str,
    model: str = "llama3.2",
) -> dict:
    """
    Generate a report section {heading, body} from raw data using the LLM.

    Args:
        topic:        Section topic (used as fallback heading if LLM fails).
        data_snippet: Raw data text for the LLM to analyse.
        model:        Ollama model name.

    Returns:
        dict with 'heading' (str) and 'body' (str). Never raises.
    """
    # TODO: ollama.chat(model=model, messages=[system, user], format='json')
    # TODO: system: 'professional report writer; return JSON {heading, body}'
    # TODO: user: f'Topic: {topic}\n\nData:\n{data_snippet[:500]}'
    # TODO: try: result = json.loads(response['message']['content'])
    # TODO:      return {'heading': str(result.get('heading', topic)),
    # TODO:              'body':    str(result.get('body', ''))}
    # TODO: except Exception: return {'heading': topic, 'body': raw}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'ai_generate_section' in globals()
        passed += 1; print('\u2705 Check 1: ai_generate_section defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None

    # Check 2: returns a dict (1 LLM call)
    try:
        result = ai_generate_section(
            'Sales Performance',
            'Q1: 1.2M revenue, up 15%% YoY. Top product: Widget A.',
        )
        assert isinstance(result, dict), f'expected dict, got {type(result)}'
        passed += 1; print('\u2705 Check 2: returns a dict')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: dict has 'heading' key with non-empty string
    try:
        assert result is not None, 'result is None (Check 2 failed)'
        assert 'heading' in result, f"missing 'heading' key: {result}"
        assert isinstance(result['heading'], str) and len(result['heading']) > 0, \
            f"heading should be non-empty str: {result['heading']!r}"
        passed += 1; print(f"\u2705 Check 3: heading={result['heading']!r}")
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: dict has 'body' key with non-empty string
    try:
        assert result is not None, 'result is None'
        assert 'body' in result, f"missing 'body' key: {result}"
        assert isinstance(result['body'], str) and len(result['body']) > 0, \
            f"body should be non-empty str: {result['body']!r}"
        passed += 1; print(f"\u2705 Check 4: body is {len(result['body'])} chars")
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: works on a different topic (1 LLM call)
    try:
        result2 = ai_generate_section(
            'Market Overview',
            'Total market: 50B. Our share: 2.4%%. Three main competitors.',
        )
        assert isinstance(result2, dict), f'expected dict, got {type(result2)}'
        assert 'heading' in result2 and 'body' in result2
        passed += 1; print('\u2705 Check 5: works on different topic')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_generate_section(
    topic: str,
    data_snippet: str,
    model: str = "llama3.2",
) -> dict:
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional report writer. "
                    "Generate a concise report section from the data provided. "
                    'Return JSON with exactly two keys: "heading" (a short title string) '
                    'and "body" (2-4 sentences of professional analysis). '
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Topic: {topic}\n\n"
                    f"Data:\n{data_snippet[:500]}\n\n"
                    "Generate a report section:"
                ),
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        result = json.loads(raw)
        return {
            "heading": str(result.get("heading", topic)),
            "body": str(result.get("body", "")),
        }
    except Exception:
        return {"heading": topic, "body": raw}
```

</details>